# 🤖 Gemini API 대량 분석 노트북

## 개요

이 노트북은 **텍스트 또는 이미지 리스트**를 Gemini API로 대량 분석하는 전체 워크플로를 다룹니다.

### 전체 흐름

```
[Excel 파일] 
    ↓ 데이터 읽기
[텍스트 / 이미지 리스트]
    ↓ Gemini API 순차 호출
[분석 결과 (분류/요약 등)]
    ↓ JSON 저장
[results.json]
    ↓ 원본 Excel에 열 추가
[최종 Excel 파일]
```

### 필요 패키지

| 패키지 | 용도 |
|--------|------|
| `google-genai` | Gemini API 호출 |
| `openpyxl` | Excel 읽기/쓰기 |
| `python-dotenv` | `.env`에서 API Key 로드 |
| `Pillow` | 이미지 처리 (표준 패키지, 별도 설치 필요) |

```bash
pip install google-genai openpyxl python-dotenv Pillow
```

> 💡 **이미지 분석**은 `Pillow` 하나만 추가하면 됩니다. base64로 인코딩해 API에 전달하는 방식이라 별도 복잡한 설정이 없습니다.

## STEP 0 : 환경 설정

`.env` 파일에 API 키를 저장합니다. 노트북과 **같은 폴더**에 `.env` 파일을 만들고 아래 내용을 입력하세요:

```
GEMINI_API_KEY=여기에_본인_API_키_입력
```

> ⚠️ `.env` 파일은 절대 Git에 올리지 마세요! `.gitignore`에 추가하세요.

In [8]:
# ─────────────────────────────────────────
# 패키지 임포트
# ─────────────────────────────────────────
import os
import json
import time
import base64
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 불러오기
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# API 키가 제대로 로드됐는지 확인 (키 앞 5자리만 출력)
if GEMINI_API_KEY:
    print(f"✅ API Key 로드 성공: {GEMINI_API_KEY[:5]}...")
else:
    print("❌ API Key를 찾을 수 없습니다. .env 파일을 확인하세요.")

✅ API Key 로드 성공: AIzaS...


In [9]:
# ─────────────────────────────────────────
# Gemini 클라이언트 초기화
# ─────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

# 사용할 모델 지정 (gemini-2.5-flash-lite: 빠르고 저렴)
MODEL = "gemini-2.5-flash-lite"

print(f"✅ Gemini 클라이언트 초기화 완료. 사용 모델: {MODEL}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Gemini 클라이언트 초기화 완료. 사용 모델: gemini-2.5-flash-lite


## STEP 1 : 프롬프트 작성

분석 목적에 맞게 **프롬프트 템플릿**을 작성합니다.

- `{text}` 자리에 각 항목의 텍스트가 자동으로 삽입됩니다.
- 결과는 반드시 **JSON 형식**으로 반환하도록 지시합니다 (후처리 편의를 위해).
- 아래는 예시이며, **원하는 분석으로 자유롭게 수정**하세요.

In [11]:
# ─────────────────────────────────────────
# ✏️  여기를 수정하여 원하는 분석 프롬프트를 작성하세요
# ─────────────────────────────────────────

PROMPT_TEMPLATE = """
다음 텍스트를 분석하고, 아래 JSON 형식으로만 응답하세요. 다른 설명은 절대 추가하지 마세요.

[텍스트]
{text}

[응답 형식]
{{
  "category": "긍정|부정|중립 중 하나",
  "summary": "한 문장 요약",
  "keywords": ["핵심어1", "핵심어2", "핵심어3"],
  "confidence": 0.0~1.0 사이 숫자
}}
"""

# 이미지 분석용 프롬프트 (이미지는 {text} 없이 사용)
IMAGE_PROMPT = """
이 이미지를 분석하고, 아래 JSON 형식으로만 응답하세요. 다른 설명은 절대 추가하지 마세요.

[응답 형식]
{{
  "description": "이미지 내용 설명",
  "objects": ["객체1", "객체2"],
  "sentiment": "긍정|부정|중립 중 하나",
  "confidence": 0.0~1.0 사이 숫자
}}
"""

print("✅ 프롬프트 설정 완료")
print("\n현재 프롬프트 템플릿:")
print(PROMPT_TEMPLATE)

✅ 프롬프트 설정 완료

현재 프롬프트 템플릿:

다음 텍스트를 분석하고, 아래 JSON 형식으로만 응답하세요. 다른 설명은 절대 추가하지 마세요.

[텍스트]
{text}

[응답 형식]
{{
  "category": "긍정|부정|중립 중 하나",
  "summary": "한 문장 요약",
  "keywords": ["핵심어1", "핵심어2", "핵심어3"],
  "confidence": 0.0~1.0 사이 숫자
}}



## STEP 2 : API 호출 함수 정의

텍스트와 이미지 각각에 대한 API 호출 함수를 정의합니다.

### 이미지 처리 방식
- 이미지 파일을 **base64로 인코딩**해 API에 전달합니다.
- `Pillow` 패키지로 이미지를 열어 MIME 타입을 판별합니다.
- 별도의 복잡한 설정 없이 `Pillow` 하나로 충분합니다.

In [ ]:
# ─────────────────────────────────────────
# 텍스트 분석 함수
# ─────────────────────────────────────────
def analyze_text(text: str, prompt_template: str = PROMPT_TEMPLATE) -> dict:
    """
    텍스트 하나를 Gemini API로 분석합니다.
    
    Args:
        text: 분석할 텍스트
        prompt_template: {text}를 포함한 프롬프트 템플릿
    
    Returns:
        dict: 분석 결과 (JSON 파싱됨)
    """
    try:
        # 프롬프트에 텍스트 삽입
        prompt = prompt_template.format(text=text)
        
        # Gemini API 호출
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        
        # 응답 텍스트에서 JSON 파싱
        raw = response.text.strip()
        
        # 마크다운 코드블록 제거 (```json ... ``` 형태 대응)
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        
        result = json.loads(raw)
        result["_status"] = "success"
        return result
    
    except json.JSONDecodeError:
        # JSON 파싱 실패 시 원본 텍스트를 저장
        return {"_status": "json_error", "_raw": response.text}
    
    except Exception as e:
        # 기타 오류 (네트워크, API 한도 등)
        return {"_status": "error", "_error": str(e)}


# ─────────────────────────────────────────
# 이미지 분석 함수 (로컬 파일 + URL 지원)
# ─────────────────────────────────────────
def _guess_mime_from_url(url: str) -> str:
    """URL 경로에서 확장자를 추출해 MIME 타입을 추측합니다."""
    from urllib.parse import urlparse
    path = urlparse(url).path.lower()
    # 마지막 확장자 우선 (.jpg.webp → webp)
    ext_map = {
        ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".png": "image/png",  ".webp": "image/webp",
        ".gif": "image/gif",  ".bmp": "image/bmp",
        ".tiff": "image/tiff", ".tif": "image/tiff",
    }
    for ext, mime in ext_map.items():
        if path.endswith(ext):
            return mime
    return ""


def analyze_image(image_source: str, prompt: str = IMAGE_PROMPT) -> dict:
    """
    이미지를 Gemini API로 분석합니다.
    로컬 파일 경로 또는 URL을 모두 지원합니다.
    jpg, png, webp, gif 등 주요 이미지 형식을 지원합니다.
    
    Args:
        image_source: 이미지 파일 경로 또는 URL
        prompt: 분석 지시 프롬프트
    
    Returns:
        dict: 분석 결과 (JSON 파싱됨)
    """
    try:
        from PIL import Image
        import io
        import urllib.request
        
        is_url = image_source.startswith(("http://", "https://"))
        
        # URL인지 로컬 파일인지 판별
        if is_url:
            # URL에서 이미지 다운로드
            req = urllib.request.Request(image_source, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                raw_bytes = resp.read()
                content_type = resp.headers.get("Content-Type", "")
            img = Image.open(io.BytesIO(raw_bytes))
        else:
            # 로컬 파일 읽기
            img = Image.open(image_source)
            content_type = ""
            raw_bytes = None
        
        # MIME 타입 결정 (3단계 폴백)
        # 1순위: Pillow가 감지한 포맷
        # 2순위: URL 확장자 / HTTP Content-Type
        # 3순위: JPEG 기본값
        fmt = img.format  # Pillow가 감지한 포맷 (예: "WEBP", "JPEG", "PNG")
        mime_map = {
            "PNG": "image/png", "JPEG": "image/jpeg",
            "WEBP": "image/webp", "GIF": "image/gif",
            "BMP": "image/bmp", "TIFF": "image/tiff",
        }
        
        if fmt and fmt in mime_map:
            mime_type = mime_map[fmt]
        elif is_url:
            # URL 확장자에서 추측
            mime_type = _guess_mime_from_url(image_source)
            if not mime_type and "image/" in content_type:
                mime_type = content_type.split(";")[0].strip()
            if not mime_type:
                mime_type = "image/jpeg"
            # fmt도 맞춰줌
            fmt = {v: k for k, v in mime_map.items()}.get(mime_type, "JPEG")
        else:
            mime_type = "image/jpeg"
            fmt = "JPEG"
        
        # URL에서 다운로드한 원본 바이트를 그대로 사용 (재인코딩 손실 방지)
        if is_url and raw_bytes:
            img_bytes = raw_bytes
        else:
            buf = io.BytesIO()
            # RGBA/P 모드 이미지를 JPEG로 저장 시 RGB 변환 필요
            if fmt == "JPEG" and img.mode in ("RGBA", "P"):
                img = img.convert("RGB")
            img.save(buf, format=fmt)
            img_bytes = buf.getvalue()
        
        # Gemini API 호출 (텍스트 + 이미지 멀티모달)
        response = client.models.generate_content(
            model=MODEL,
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type=mime_type),
                prompt
            ]
        )
        
        # JSON 파싱
        raw = response.text.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        
        result = json.loads(raw)
        result["_status"] = "success"
        result["_image"] = str(image_source)
        return result
    
    except json.JSONDecodeError:
        return {"_status": "json_error", "_raw": response.text, "_image": str(image_source)}
    
    except Exception as e:
        return {"_status": "error", "_error": str(e), "_image": str(image_source)}


print("✅ API 호출 함수 정의 완료")

## STEP 3 : Excel에서 데이터 읽기

분석할 데이터가 담긴 Excel 파일을 읽습니다.

- `EXCEL_FILE`: 분석할 Excel 파일 경로
- `TEXT_COLUMN`: 분석할 텍스트가 있는 열 이름
- `IMAGE_COLUMN`: (선택) 이미지 파일 경로가 있는 열 이름

In [13]:
# ─────────────────────────────────────────
# ✏️  여기를 수정하세요: 파일 경로와 열 이름
# ─────────────────────────────────────────

EXCEL_FILE   = "data.xlsx"     # 분석할 Excel 파일명
TEXT_COLUMN  = "content"       # 텍스트가 있는 열 이름
IMAGE_COLUMN = None            # 이미지 경로 열 이름 (없으면 None)
ID_COLUMN    = None            # 고유 ID 열 이름 (없으면 None → 인덱스 사용)

OUTPUT_JSON  = "results.json"  # 결과 저장 JSON 파일명
OUTPUT_EXCEL = "results_merged.xlsx"  # 최종 병합 Excel 파일명

# ─────────────────────────────────────────
# Excel 파일 읽기
# ─────────────────────────────────────────

# 파일이 없으면 샘플 데이터로 대체
if not Path(EXCEL_FILE).exists():
    print(f"⚠️  '{EXCEL_FILE}' 파일이 없어 샘플 데이터를 사용합니다.")
    df = pd.DataFrame({
        "id": [1, 2, 3, 4, 5],
        "content": [
            "이 제품은 정말 훌륭합니다! 품질도 좋고 배송도 빨랐어요.",
            "기대했던 것보다 많이 실망스럽네요. 품질이 너무 떨어집니다.",
            "그냥 평범한 제품입니다. 가격 대비 나쁘지는 않아요.",
            "완전 최고! 강력 추천합니다. 다음에도 꼭 구매할 예정입니다.",
            "배송이 늦어서 불편했지만 제품 자체는 괜찮았습니다."
        ]
    })
    # 샘플 Excel 파일 저장
    df.to_excel(EXCEL_FILE, index=False)
    print(f"   → 샘플 '{EXCEL_FILE}' 파일을 생성했습니다.")
    TEXT_COLUMN = "content"
    ID_COLUMN   = "id"
else:
    df = pd.read_excel(EXCEL_FILE)
    print(f"✅ '{EXCEL_FILE}' 파일 읽기 완료")

print(f"\n📊 데이터 크기: {len(df)} 행 × {len(df.columns)} 열")
print(f"📋 열 목록: {list(df.columns)}")
df.head()

✅ 'data.xlsx' 파일 읽기 완료

📊 데이터 크기: 5 행 × 2 열
📋 열 목록: ['id', 'content']


,id,content
0,1,이 제품은 정말 훌륭합니다! 품질도 좋고 배송도 빨랐어요.
1,2,기대했던 것보다 많이 실망스럽네요. 품질이 너무 떨어집니다.
2,3,그냥 평범한 제품입니다. 가격 대비 나쁘지는 않아요.
3,4,완전 최고! 강력 추천합니다. 다음에도 꼭 구매할 예정입니다.
4,5,배송이 늦어서 불편했지만 제품 자체는 괜찮았습니다.


## STEP 4 : 대량 분석 실행 (텍스트)

Excel에서 읽어온 텍스트 리스트를 **하나씩 순회**하며 Gemini API를 호출합니다.

### 핵심 전략
- **API 호출 간격**: 과도한 요청을 막기 위해 각 호출 사이에 잠깐 대기합니다 (Rate Limit 방지)
- **오류 건너뛰기**: 하나가 실패해도 전체 작업이 멈추지 않습니다
- **중간 저장**: 진행 중간중간 JSON으로 저장해 중단 시 손실을 줄입니다

In [14]:
# ─────────────────────────────────────────
# 대량 텍스트 분석 실행
# ─────────────────────────────────────────

DELAY_SECONDS = 0.5   # API 호출 간 대기 시간 (초) - Rate Limit 방지
SAVE_EVERY    = 10    # 몇 건마다 중간 저장할지

results = []  # 분석 결과 리스트

for i, row in df.iterrows():
    # ID 설정 (지정 열이 있으면 그 값, 없으면 인덱스)
    item_id = row[ID_COLUMN] if ID_COLUMN else i
    
    # 분석할 텍스트 가져오기
    text = str(row[TEXT_COLUMN])
    
    print(f"[{i+1}/{len(df)}] ID={item_id} 분석 중... ", end="", flush=True)
    
    # Gemini API 호출
    result = analyze_text(text)
    
    # ID와 원본 텍스트 앞에 붙이기 (나중에 Excel 병합 시 사용)
    result["_id"]   = item_id
    result["_index"] = i
    result["_text"] = text[:100] + "..." if len(text) > 100 else text  # 앞 100자만 저장
    
    results.append(result)
    
    # 성공/실패 표시
    status_icon = "✅" if result["_status"] == "success" else "❌"
    print(f"{status_icon} ({result['_status']})")
    
    # SAVE_EVERY건마다 중간 저장
    if (i + 1) % SAVE_EVERY == 0:
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"   💾 중간 저장: {len(results)}건 → {OUTPUT_JSON}")
    
    # API 호출 간 대기 (Rate Limit 방지)
    time.sleep(DELAY_SECONDS)

# ─────────────────────────────────────────
# 최종 JSON 저장
# ─────────────────────────────────────────
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n🎉 분석 완료!")
print(f"   총 {len(results)}건 처리")
print(f"   성공: {sum(1 for r in results if r['_status']=='success')}건")
print(f"   실패: {sum(1 for r in results if r['_status']!='success')}건")
print(f"   결과 저장: {OUTPUT_JSON}")

[1/5] ID=0 분석 중... ✅ (success)
[2/5] ID=1 분석 중... ✅ (success)
[3/5] ID=2 분석 중... ✅ (success)
[4/5] ID=3 분석 중... ✅ (success)
[5/5] ID=4 분석 중... ✅ (success)

🎉 분석 완료!
   총 5건 처리
   성공: 5건
   실패: 0건
   결과 저장: results.json


## STEP 4-B : 이미지 대량 분석 (선택)

Excel의 특정 열에 이미지 파일 경로가 있을 경우 이미지 분석을 실행합니다.

- `IMAGE_COLUMN`이 설정된 경우: Excel에서 이미지 경로/URL을 읽어 분석
- `IMAGE_COLUMN`이 없는 경우: 샘플 URL 리스트로 분석 실행
- 텍스트 분석과 완전히 동일한 패턴입니다.

In [ ]:
# ─────────────────────────────────────────
# 이미지 대량 분석
# IMAGE_COLUMN이 설정된 경우 → Excel에서 이미지 경로/URL 읽기
# IMAGE_COLUMN이 없는 경우  → 샘플 URL 리스트로 분석
# ─────────────────────────────────────────

# Excel에 이미지 경로 열이 있으면 그것을 사용, 없으면 샘플 URL 리스트 사용
if IMAGE_COLUMN and IMAGE_COLUMN in df.columns:
    image_list = df[IMAGE_COLUMN].astype(str).tolist()
    id_list = (df[ID_COLUMN] if ID_COLUMN else df.index).tolist()
    print(f"🖼️  이미지 분석 시작 (Excel 열: '{IMAGE_COLUMN}', {len(image_list)}건)")
else:
    print(f"⚠️  IMAGE_COLUMN이 설정되지 않아 샘플 이미지 URL 리스트를 사용합니다.")
    image_list = [
        "https://assetmanagerpim-res.cloudinary.com/images/2efce93c613d440a96d34360cff3b119_9366/JN0726_000_plp_model.jpg",
        "https://www.ilyosisa.co.kr/data/photos/20231148/art_17012271521182_a5f7b2.jpg",
        "https://dimg.donga.com/wps/NEWS/IMAGE/2019/04/24/95195044.1.jpg",
        "https://newsx.ecn.cdn.infralab.net/news.eduhope.net/imgdata/news_eduhope_net/202210/2022100744153797.jpg",
        "https://ichef.bbci.co.uk/ace/ws/800/cpsprodpb/10cd/live/b5bfdcc0-08ef-11ee-b5af-25e80c61c11a.jpg.webp",
    ]
    id_list = list(range(len(image_list)))
    print(f"🖼️  이미지 분석 시작 (샘플 URL {len(image_list)}건)")

image_results = []

for i, (item_id, image_source) in enumerate(zip(id_list, image_list)):
    print(f"[{i+1}/{len(image_list)}] ID={item_id} 이미지 분석 중... ", end="", flush=True)
    
    # URL 또는 로컬 파일 경로 모두 지원
    if not image_source.startswith(("http://", "https://")) and not Path(image_source).exists():
        result = {"_status": "error", "_error": f"파일 없음: {image_source}"}
    else:
        result = analyze_image(image_source)
    
    result["_id"]    = item_id
    result["_index"] = i
    image_results.append(result)
    
    status_icon = "✅" if result["_status"] == "success" else "❌"
    print(f"{status_icon}")
    
    time.sleep(DELAY_SECONDS)

# 이미지 결과 저장
image_json = "image_results.json"
with open(image_json, "w", encoding="utf-8") as f:
    json.dump(image_results, f, ensure_ascii=False, indent=2)

print(f"\n🎉 이미지 분석 완료!")
print(f"   총 {len(image_results)}건 처리")
print(f"   성공: {sum(1 for r in image_results if r['_status']=='success')}건")
print(f"   실패: {sum(1 for r in image_results if r['_status']!='success')}건")
print(f"   결과 저장: {image_json}")

## STEP 5 : JSON 결과를 원본 Excel에 병합

저장된 `results.json`을 읽어 **원본 Excel에 열을 추가**한 새 파일을 만듭니다.

### 작동 방식
1. `results.json` 로드 → DataFrame 변환
2. 원본 Excel DataFrame과 `_index` 기준으로 병합(join)
3. 불필요한 내부 컬럼(`_` 접두사) 정리
4. 새 Excel 파일로 저장

In [ ]:
# ─────────────────────────────────────────
# JSON 결과 로드
# ─────────────────────────────────────────
with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print(f"✅ {OUTPUT_JSON} 로드 완료: {len(saved_results)}건")

# JSON 리스트 → DataFrame 변환
df_results = pd.DataFrame(saved_results)

print(f"\n📋 결과 DataFrame 열 목록: {list(df_results.columns)}")
df_results.head()

In [ ]:
# ─────────────────────────────────────────
# 원본 Excel과 결과 DataFrame 병합
# ─────────────────────────────────────────

# 원본 DataFrame에 인덱스 열 추가 (병합 기준)
df_original = df.copy()
df_original["_index"] = df_original.index

# _index 기준으로 LEFT JOIN
df_merged = df_original.merge(
    df_results,
    on="_index",        # 인덱스 기준 병합
    how="left",         # 원본 행은 모두 유지
    suffixes=("", "_result")  # 열 이름 충돌 시 구분
)

# 내부 관리용 컬럼 제거 (사용자에게 필요 없는 것들)
cols_to_drop = [c for c in df_merged.columns if c in ["_index", "_id", "_text"]]
df_merged = df_merged.drop(columns=cols_to_drop)

print(f"✅ 병합 완료")
print(f"   원본 열: {list(df.columns)}")
print(f"   추가된 열: {[c for c in df_merged.columns if c not in df.columns]}")
print(f"\n병합 결과 미리보기:")
df_merged.head()

In [ ]:
# ─────────────────────────────────────────
# 최종 Excel 파일로 저장
# ─────────────────────────────────────────

# keywords 열이 리스트일 경우 문자열로 변환 (Excel 호환)
for col in df_merged.columns:
    if df_merged[col].apply(lambda x: isinstance(x, list)).any():
        df_merged[col] = df_merged[col].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else x
        )

# Excel 저장
df_merged.to_excel(OUTPUT_EXCEL, index=False)

print(f"🎉 최종 파일 저장 완료: {OUTPUT_EXCEL}")
print(f"   행: {len(df_merged)}행 / 열: {len(df_merged.columns)}개")
print(f"\n💡 {OUTPUT_EXCEL} 파일을 열어 결과를 확인하세요!")

## STEP 6 : 결과 요약 확인

분석 결과를 간단하게 요약해서 확인합니다.

In [ ]:
# ─────────────────────────────────────────
# 결과 요약 통계
# ─────────────────────────────────────────

print("="*50)
print("📊 분석 결과 요약")
print("="*50)

# 성공/실패 현황
if "_status" in df_merged.columns:
    status_counts = df_merged["_status"].value_counts()
    print("\n[처리 현황]")
    for status, count in status_counts.items():
        print(f"  {status}: {count}건")

# 분류 결과 분포 (category 열이 있을 때)
if "category" in df_merged.columns:
    print("\n[분류 결과 분포]")
    category_counts = df_merged["category"].value_counts()
    for cat, count in category_counts.items():
        bar = "█" * count
        print(f"  {cat:8s}: {bar} ({count}건)")

# 신뢰도 평균 (confidence 열이 있을 때)
if "confidence" in df_merged.columns:
    avg_conf = pd.to_numeric(df_merged["confidence"], errors="coerce").mean()
    print(f"\n[평균 신뢰도]  {avg_conf:.3f}")

print("\n" + "="*50)
print(f"✅ 전체 파이프라인 완료!")
print(f"   JSON 결과: {OUTPUT_JSON}")
print(f"   Excel 결과: {OUTPUT_EXCEL}")

---
## 📌 자주 묻는 질문

### Q1. 프롬프트를 바꾸고 싶어요
STEP 1의 `PROMPT_TEMPLATE`을 수정하세요. `{text}` 위치에 각 텍스트가 삽입됩니다. JSON 응답 형식을 유지해야 STEP 5 병합이 잘 됩니다.

### Q2. API 요청이 너무 느려요
`DELAY_SECONDS = 0.5`를 줄이거나 `0`으로 설정하세요. 단, Rate Limit 오류가 날 수 있습니다.

### Q3. 중간에 오류가 나면 처음부터 다시 해야 하나요?
`SAVE_EVERY`마다 JSON에 저장하므로 중간 저장된 결과를 활용할 수 있습니다.  
또는 `results.json`에서 `_status == 'error'`인 건만 골라 재시도하세요.

### Q4. 이미지 분석에 필요한 것이 Pillow뿐인가요?
네. `pip install Pillow` 하나면 됩니다. 이미지를 base64로 인코딩해서 API에 전달하는 방식이라 추가 패키지가 필요 없습니다.

### Q5. JSON이 아닌 다른 형식으로 받고 싶어요
프롬프트에서 JSON 형식 요구를 제거하고, `analyze_text()` 함수에서 `json.loads()` 부분을 `response.text` 그대로 반환하도록 수정하세요.